# Prática 3 — Do mapa do diretório ao impacto

Esta prática refaz, com `pandas`, a caçada que a Seção~1.7 do capítulo descreve no Kibana.
Cada passo mostra primeiro a **consulta ingênua**, que cai na isca, e depois a **pergunta refinada**,
que responde à hipótese. No fim, as respostas são conferidas contra o gabarito que o gerador do
cenário recalcula por código.

**Hipótese.** Antes de cifrar, o adversário mapeia o diretório: uma conta consulta muitos grupos diferentes em pouco tempo, e depois se move para os servidores de arquivos.

## Carregar a evidência

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))
import pandas as pd

from eventos import conferir, ler_eventos, ler_gabarito

eventos = ler_eventos()
gabarito = ler_gabarito()
print(f"{len(eventos)} eventos, de {eventos['@timestamp'].min()} a {eventos['@timestamp'].max()}")
eventos["event.dataset"].value_counts()

49277 eventos, de 2026-03-09 00:00:07+00:00 a 2026-03-11 23:59:42+00:00


event.dataset
dns           22159
firewall       9655
fileserver     7340
edr            4696
vpn            2967
auth           2460
Name: count, dtype: int64

## Passo 1 — quem mais enumera, e por que engana

```
event.dataset:"auth" and event.code:"4799"  →  contagem por user.name
```

In [2]:
enumeracoes = eventos[(eventos["event.dataset"] == "auth") & (eventos["event.code"] == "4799")]
enumeracoes["user.name"].value_counts().head(5)

user.name
svc_inventario    1296
adm.backup          61
Name: count, dtype: int64

O topo é o inventário, que roda de dez em dez minutos sobre **os mesmos poucos grupos**.

In [3]:
enumeracoes.groupby("user.name").agg(eventos=("group.name", "size"),
                                     grupos=("group.name", "nunique")).sort_values("grupos", ascending=False).head(5)

,eventos,grupos
user.name,,
adm.backup,61,47
svc_inventario,1296,3


## Passo 2 — amplitude em vez de volume

In [4]:
por_conta = enumeracoes.groupby("user.name")["group.name"].nunique().sort_values(ascending=False)
conta = por_conta.index[0]
grupos = int(por_conta.iloc[0])
print(conta, grupos)

adm.backup 47


## Passo 3 — do endereço à estação

In [5]:
auth = eventos[eventos["event.dataset"] == "auth"]
origem = enumeracoes[enumeracoes["user.name"] == conta]["source.ip"].iloc[0]
interativos = auth[(auth["host.ip"] == origem) & (auth["winlog.logon.type"] == "2")]
estacao = interativos.iloc[0]["host.name"]
print(origem, estacao)

10.40.0.140 WKS-ADM-104


## Passo 4 — a extensão nova, e a rotina que também renomeia

O agente de backup renomeia centenas de arquivos toda madrugada. O que separa o ataque é a extensão.

In [6]:
arquivos = eventos[eventos["event.dataset"] == "fileserver"]
arquivos.groupby("file.extension").agg(renomeacoes=("process.name", "size"),
                                      processos=("process.name", "nunique"))

,renomeacoes,processos
file.extension,,
aurora-lock,3420,1
bak,1698,1
xlsx,2222,1


## Passo 5 — o primeiro servidor, o processo e o tamanho do estrago

In [7]:
cifrados = arquivos[arquivos["file.extension"] == "aurora-lock"].sort_values("@timestamp")
primeiro_servidor = cifrados.iloc[0]["host.name"]
processo = cifrados["process.name"].iloc[0]
total = len(cifrados)
print(primeiro_servidor, processo, total)
cifrados.groupby("host.name")["@timestamp"].agg(["min", "max", "size"])

FS01 aurora_upd.exe 3420


,min,max,size
host.name,,,
FS01,2026-03-10 17:05:45+00:00,2026-03-10 17:30:42.300000+00:00,2140
FS02,2026-03-10 17:14:45+00:00,2026-03-10 17:29:40.300000+00:00,1280


## Passo 6 — sem volta atrás

In [8]:
edr = eventos[eventos["event.dataset"] == "edr"]
filhos = edr[edr["process.parent.name"] == processo]
comando = filhos.iloc[0]["process.command_line"]
print(comando)

vssadmin delete shadows /all /quiet


## Conferência contra o gabarito

As respostas do notebook precisam bater com as que o gerador recalcula por código sobre todos os eventos.

In [9]:
for chave, obtido in [("c3-conta", conta), ("c3-grupos", grupos), ("c3-estacao", estacao),
                      ("c3-primeiro", primeiro_servidor), ("c3-processo", processo),
                      ("c3-sombras", comando), ("c3-arquivos", total)]:
    print(conferir(chave, obtido, gabarito))

ok   c3-conta: obtido 'adm.backup', gabarito 'adm.backup'
ok   c3-grupos: obtido 47, gabarito '47'
ok   c3-estacao: obtido 'WKS-ADM-104', gabarito 'WKS-ADM-104'
ok   c3-primeiro: obtido 'FS01', gabarito 'FS01'
ok   c3-processo: obtido 'aurora_upd.exe', gabarito 'aurora_upd.exe'
ok   c3-sombras: obtido 'vssadmin delete shadows /all /quiet', gabarito 'vssadmin delete shadows /all /quiet'
ok   c3-arquivos: obtido 3420, gabarito '3420'


## Exercício

A detecção derivada dispara com 50 renomeações da mesma extensão em cinco minutos por um processo. Meça, nos dados, quantas vezes o agente de backup dispararia essa regra, e ajuste o limiar.